In [0]:
movies_path = "/Volumes/movielakehouse/bronze/raw/tmdb_5000_movies.csv"
credits_path = "/Volumes/movielakehouse/bronze/raw/tmdb_5000_credits.csv"

display(dbutils.fs.ls("/Volumes/movielakehouse/bronze/raw/"))

In [0]:
movies_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(movies_path)
)

credits_df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(credits_path)
)

In [0]:
display(movies_df.limit(2))
display(credits_df.limit(2))

In [0]:
print("********MOVIES********")
movies_df.printSchema()
print(f"Rows: {movies_df.count():,}")
print(f"Columns: {len(movies_df.columns)}")
print(movies_df.columns)

print("\n********CREDITS********")
credits_df.printSchema()
print(f"Rows: {credits_df.count():,}")
print(f"Columns: {len(credits_df.columns)}")
print(credits_df.columns)

In [0]:
# chaves, nulidade, duplicidade e relacionamento

from pyspark.sql import functions as F


def null_profile(df):
    return df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    )

print("****** NULL MOVIES *******")
display(null_profile(movies_df))

print("****** NULL CREDITS *******")
display(null_profile(credits_df))

In [0]:
# testar movies.id e credits.movie_id como chaves candidatas

movies_keys_stats = movies_df.agg(
    F.count("*").alias("rows"),
    F.count("id").alias("non_null_id"),
    F.countDistinct("id").alias("distinct_id"),
)

credits_keys_stats = credits_df.agg(
    F.count("*").alias("rows"),
    F.count("movie_id").alias("non_null_movie_id"),
    F.countDistinct("movie_id").alias("distinct_movie_id"),
)

display(movies_keys_stats)
display(credits_keys_stats)

In [0]:
# Verificar duplicidades concretas

display(
    movies_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(
    credits_df
    .groupBy("movie_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

In [0]:
# Testar a covertura entre as fontes

movies_without_credits = (
    movies_df
    .select(F.col("id").alias('movie_id'))
    .join(
        credits_df.select("movie_id"),"movie_id","left_anti"
    )
)

credits_without_movies = (
    credits_df
    .select('movie_id')
    .join(
        movies_df.select(F.col("id").alias("movie_id")), "movie_id", "left_anti"
    )
)

print(movies_without_credits.count())
print(credits_without_movies.count())

In [0]:
# Verificar divergências entre os títulos em ambos df

titles_mismatches = (
    movies_df.alias("m")
    .join(
        credits_df.alias("c"),
        F.col("m.id") == F.col("c.movie_id"),
        "inner"
    )
    .filter(
        ~F.col("m.title").eqNullSafe(F.col("c.title"))
    )
    .select(
        F.col("m.id").alias("movie_id"),
        F.col("m.title").alias("movie_title"),
        F.col("c.title").alias("credits_title")
    )
)

print(f"Divergências de title: {titles_mismatches.count()}")
display(titles_mismatches.limit(20))

In [0]:
# Distringuir null d evalores que representem ausência
display(
    movies_df.agg(
        F.sum((F.col("budget") == 0).cast("int")).alias("budget_zero"),
        F.sum(F.col("budget").isNull().cast("int")).alias("budget_null"),
        F.sum((F.col("revenue") == 0).cast("int")).alias("revenue_zero"),
        F.sum(F.col("revenue").isNull().cast("int")).alias("revenue_null"),
        F.sum(F.col("release_date").isNull().cast("int")).alias("release_date_null"),
        F.sum((F.col("vote_count") == 0).cast("int")).alias("vote_count_zero"),
        F.sum(F.col("vote_count").isNull().cast("int")).alias("vote_count_null")
    )
)

In [0]:
# ntervalos observados

display(
    movies_df.agg(
        F.min("release_date").alias("min_release_date"),
        F.max("release_date").alias("max_release_date"),
        F.min("budget").alias("min_budget"),
        F.max("budget").alias("max_budget"),
        F.min("revenue").alias("min_revenue"),
        F.max("revenue").alias("max_revenue"),
        F.min("popularity").alias("min_popularity"),
        F.max("popularity").alias("max_popularity"),
        F.min("vote_average").alias("min_vote_average"),
        F.max("vote_average").alias("max_vote_average"),
        F.min("vote_count").alias("min_vote_count"),
        F.max("vote_count").alias("max_vote_count")
    )
)

In [0]:
# Investigando Estruturas semiestruturadas

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType

id_name_schema = ArrayType(
    StructType([
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True)
    ])
)

cast_schema = ArrayType(
    StructType([
        StructField("cast_id", IntegerType(), True),
        StructField("character", StringType(), True),
        StructField("credit_id", StringType(), True),
        StructField("gender", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("order", IntegerType(), True)
    ])
)

crew_schema = ArrayType(
    StructType([
        StructField("credit_id", StringType(), True),
        StructField("department", StringType(), True),
        StructField("gender", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("job", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

In [0]:
movies_parsed = movies_df.withColumn(
    "genres_parsed", F.from_json("genres", id_name_schema)
)

credits_parsed = credits_df.withColumn(
    "cast_parsed", F.from_json("cast", cast_schema)
).withColumn("crew_parsed", F.from_json("crew", crew_schema))


display(
    movies_parsed.agg(
        F.sum(F.col("genres_parsed").isNull().cast("int")).alias("genres_parse_null"),
        F.sum((F.size("genres_parsed") == 0).cast("int")).alias("genres_empty")
    )
)

display(
    credits_parsed.agg(
        F.sum(F.col("cast_parsed").isNull().cast("int")).alias("cast_parse_null"),
        F.sum((F.size("cast_parsed") == 0).cast("int")).alias("cast_empty"),
        F.sum(F.col("crew_parsed").isNull().cast("int")).alias("crew_parse_null"),
        F.sum((F.size("crew_parsed") == 0).cast("int")).alias("crew_empty")
    )
)

In [0]:
display(movies_parsed.limit(1))
display(credits_parsed.limit(1))

In [0]:
# verificano o dominio existente

genres_exploded = (
    movies_parsed
    .select("id", F.explode("genres_parsed").alias("genre"))
)

display(
    genres_exploded
    .groupBy("genre.id", "genre.name")
    .count()
    .orderBy(F.desc("count"))
)

crew_exploded = (
    credits_parsed
    .select("movie_id", F.explode("crew_parsed").alias("member"))
)

display(
    crew_exploded
    .groupBy("member.department")
    .count()
    .orderBy(F.desc("count"))
)

display(
    crew_exploded
    .groupBy("member.job")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

In [0]:
display(
    movies_parsed.agg(
        F.sum(F.size("genres_parsed")).alias("genre_relations"),
        F.max(F.size("genres_parsed")).alias("max_genres_per_movie")
    )
)

display(
    credits_parsed.agg(
        F.sum(F.size("cast_parsed")).alias("cast_relations"),
        F.max(F.size("cast_parsed")).alias("max_cast_per_movie"),
        F.sum(F.size("crew_parsed")).alias("crew_relations"),
        F.max(F.size("crew_parsed")).alias("max_crew_per_movie")
    )
)

In [0]:
iso_country_schema = ArrayType(
    StructType([
        StructField("iso_3166_1", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

iso_language_schema = ArrayType(
    StructType([
        StructField("iso_639_1", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

semi_structured = (
    movies_df
    .withColumn("keywords_p", F.from_json("keywords", id_name_schema))
    .withColumn("companies_p", F.from_json("production_companies", id_name_schema))
    .withColumn("countries_p", F.from_json("production_countries", iso_country_schema))
    .withColumn("languages_p", F.from_json("spoken_languages", iso_language_schema))
)

display(
    semi_structured.agg(
        *[
            expr
            for c in ["keywords_p", "companies_p", "countries_p", "languages_p"]
            for expr in (
                F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_parse_null"),
                F.sum((F.size(c) == 0).cast("int")).alias(f"{c}_empty")
            )
        ]
    )
)

In [0]:
display(
    movies_df.agg(
        F.sum(((F.col("budget") > 0) & (F.col("revenue") > 0)).cast("int"))
            .alias("budget_and_revenue_positive"),
        F.sum(((F.col("budget") == 0) & (F.col("revenue") > 0)).cast("int"))
            .alias("budget_zero_revenue_positive"),
        F.sum(((F.col("budget") > 0) & (F.col("revenue") == 0)).cast("int"))
            .alias("budget_positive_revenue_zero"),
        F.sum(((F.col("budget") == 0) & (F.col("revenue") == 0)).cast("int"))
            .alias("budget_and_revenue_zero")
    )
)

In [0]:
display(
    movies_df.agg(
        F.sum(((F.col("vote_count") == 0) & (F.col("vote_average") == 0)).cast("int"))
            .alias("votes_zero_average_zero"),
        F.sum(((F.col("vote_count") == 0) & (F.col("vote_average") != 0)).cast("int"))
            .alias("votes_zero_average_nonzero"),
        F.sum(((F.col("vote_count") > 0) & (F.col("vote_average") == 0)).cast("int"))
            .alias("votes_positive_average_zero")
    )
)

In [0]:
cast_exploded = (
    credits_parsed
    .select("movie_id", F.explode("cast_parsed").alias("member"))
)

display(
    genres_exploded.agg(
        F.sum(F.col("genre.id").isNull().cast("int")).alias("genre_id_null"),
        F.sum(F.col("genre.name").isNull().cast("int")).alias("genre_name_null")
    )
)

display(
    cast_exploded.agg(
        F.sum(F.col("member.id").isNull().cast("int")).alias("person_id_null"),
        F.sum(F.col("member.name").isNull().cast("int")).alias("person_name_null"),
        F.sum(F.col("member.credit_id").isNull().cast("int")).alias("credit_id_null")
    )
)

display(
    crew_exploded.agg(
        F.sum(F.col("member.id").isNull().cast("int")).alias("person_id_null"),
        F.sum(F.col("member.name").isNull().cast("int")).alias("person_name_null"),
        F.sum(F.col("member.credit_id").isNull().cast("int")).alias("credit_id_null"),
        F.sum(F.col("member.department").isNull().cast("int")).alias("department_null"),
        F.sum(F.col("member.job").isNull().cast("int")).alias("job_null")
    )
)

In [0]:
keywords_exploded = semi_structured.select("id", F.explode("keywords_p").alias("item"))

companies_exploded = semi_structured.select(
    "id", F.explode("companies_p").alias("item")
)

countries_exploded = semi_structured.select(
    "id", F.explode("countries_p").alias("item")
)

languages_exploded = semi_structured.select(
    "id", F.explode("languages_p").alias("item")
)

In [0]:
# presença dos atributos estruturantes:

display(
    keywords_exploded.agg(
        F.sum(F.col("item.id").isNull().cast("int")).alias("keyword_id_null"),
        F.sum(F.col("item.name").isNull().cast("int")).alias("keyword_name_null"),
    )
)

display(
    companies_exploded.agg(
        F.sum(F.col("item.id").isNull().cast("int")).alias("company_id_null"),
        F.sum(F.col("item.name").isNull().cast("int")).alias("company_name_null"),
    )
)

display(
    countries_exploded.agg(
        F.sum(F.col("item.iso_3166_1").isNull().cast("int")).alias("country_code_null"),
        F.sum(F.col("item.name").isNull().cast("int")).alias("country_name_null"),
    )
)

display(
    languages_exploded.agg(
        F.sum(F.col("item.iso_639_1").isNull().cast("int")).alias("language_code_null"),
        F.sum(F.col("item.name").isNull().cast("int")).alias("language_name_null"),
    )
)

In [0]:
# cardinalidade total/máxima e duplicidades intrafilme:
display(
    semi_structured.agg(
        F.sum(F.size("keywords_p")).alias("keyword_relations"),
        F.max(F.size("keywords_p")).alias("max_keywords_per_movie"),
        F.sum(F.size("companies_p")).alias("company_relations"),
        F.max(F.size("companies_p")).alias("max_companies_per_movie"),
        F.sum(F.size("countries_p")).alias("country_relations"),
        F.max(F.size("countries_p")).alias("max_countries_per_movie"),
        F.sum(F.size("languages_p")).alias("language_relations"),
        F.max(F.size("languages_p")).alias("max_languages_per_movie"),
    )
)

display(keywords_exploded.groupBy("id", "item.id").count().filter(F.col("count") > 1))

display(companies_exploded.groupBy("id", "item.id").count().filter(F.col("count") > 1))

display(
    countries_exploded.groupBy("id", "item.iso_3166_1")
    .count()
    .filter(F.col("count") > 1)
)

display(
    languages_exploded.groupBy("id", "item.iso_639_1")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(genres_exploded.groupBy("id", "genre.id").count().filter(F.col("count") > 1))

display(
    cast_exploded.groupBy("movie_id", "member.credit_id")
    .count()
    .filter(F.col("count") > 1)
)

display(
    crew_exploded.groupBy("movie_id", "member.credit_id")
    .count()
    .filter(F.col("count") > 1)
)